# Ch 12 — GAE Step-by-Step Numerical Trace

**手算 GAE** 在一段固定 trajectory 上的不同 λ 值——亲眼看 λ 是怎么调 bias-variance 的。

## Setup
- Trajectory: 5 step 短轨
- rewards = [1, 0, -1, 2, 1]，V(s_t) = [0.5, 0.6, 0.4, 0.7, 0.8, 0]（最后是 terminal）
- γ = 0.99
- λ ∈ {0, 0.5, 0.95, 1}

## 学什么
1. **λ=0**: Â = δ_t（pure TD advantage，低方差高偏差）
2. **λ=1**: Â = G_t - V(s_t)（pure MC advantage，高方差低偏差）
3. **λ=0.95**: PPO 实战推荐，中间最优

## 链回
- [[ch12-eligibility-traces]] §3.5 GAE
- [[ch13-policy-gradient]] §3.6 GAE / §3.8 PPO（用 GAE 算 advantage）
- [[_anki/ch12-gae-cards]] §E

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Fixed trajectory + value estimates
rewards = np.array([1.0, 0.0, -1.0, 2.0, 1.0])
values  = np.array([0.5, 0.6,  0.4, 0.7, 0.8, 0.0])  # V(s_0)..V(s_T), terminal V=0
gamma = 0.99
T = len(rewards)

# Compute TD residuals δ_t = r_{t+1} + γ V(s_{t+1}) - V(s_t)
deltas = rewards + gamma * values[1:] - values[:-1]
print('TD residuals δ_t:')
for t, d in enumerate(deltas):
    print(f'  δ_{t} = r_{t+1} + γV(s_{t+1}) - V(s_{t}) = {rewards[t]} + {gamma}·{values[t+1]} - {values[t]} = {d:+.4f}')

## GAE 计算（reverse-time 累积）

**Â^GAE_t = δ_t + (γλ) · Â^GAE_{t+1}** —— 从末尾倒推到 t=0。

In [ ]:
def compute_gae(deltas, gamma, lam):
    T = len(deltas)
    A = np.zeros(T)
    A[-1] = deltas[-1]
    for t in reversed(range(T-1)):
        A[t] = deltas[t] + gamma * lam * A[t+1]
    return A

print(f'{"t":>3} | {"δ_t":>9} | {"λ=0":>9} | {"λ=0.5":>9} | {"λ=0.95":>10} | {"λ=1 (MC)":>10}')
print('-' * 65)
A_data = {}
for lam in [0.0, 0.5, 0.95, 1.0]:
    A_data[lam] = compute_gae(deltas, gamma, lam)

for t in range(T):
    row = f'{t:>3} | {deltas[t]:+9.4f}'
    for lam in [0.0, 0.5, 0.95, 1.0]:
        row += f' | {A_data[lam][t]:+9.4f}'
    print(row)

## 验证 λ=0 == TD adv & λ=1 == MC adv

In [ ]:
# λ=0 case: Â_t should == δ_t exactly
print('λ=0 verification:')
print(f'  Â^GAE(λ=0) = {A_data[0.0]}')
print(f'  δ_t         = {deltas}')
print(f'  Match: {np.allclose(A_data[0.0], deltas)}')

# λ=1 case: Â_t should == G_t - V(s_t) (MC advantage)
# G_t = sum of discounted rewards from t to end
G = np.zeros(T)
G[-1] = rewards[-1]
for t in reversed(range(T-1)):
    G[t] = rewards[t] + gamma * G[t+1]
mc_adv = G - values[:-1]
print(f'\nλ=1 verification:')
print(f'  Â^GAE(λ=1) = {A_data[1.0]}')
print(f'  G_t - V(s) = {mc_adv}')
print(f'  Match: {np.allclose(A_data[1.0], mc_adv)}')

## 可视化不同 λ 的 advantage 形状

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
t_axis = np.arange(T)
for lam in [0.0, 0.5, 0.95, 1.0]:
    ax.plot(t_axis, A_data[lam], marker='o', label=f'λ={lam}')
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_xlabel('Time step t')
ax.set_ylabel('Â^GAE_t')
ax.set_title('GAE advantage estimates vs λ')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 验证方差差异（蒙特卡洛实验）

在一个 stochastic 环境上比较 λ=0 跟 λ=1 advantage 的方差。

In [ ]:
# Synthetic noisy env: same V(s), but rewards are noisy
def run_one_trajectory(true_rewards, noise_std=1.0):
    return true_rewards + np.random.randn(len(true_rewards)) * noise_std

true_r = np.array([1.0, 0.0, -1.0, 2.0, 1.0])

np.random.seed(42)
advs_lam0 = []
advs_lam1 = []
for _ in range(500):
    noisy_r = run_one_trajectory(true_r, noise_std=1.0)
    deltas_n = noisy_r + gamma * values[1:] - values[:-1]
    advs_lam0.append(compute_gae(deltas_n, gamma, 0.0))
    advs_lam1.append(compute_gae(deltas_n, gamma, 1.0))

advs_lam0 = np.array(advs_lam0)
advs_lam1 = np.array(advs_lam1)

print('Variance of Â at each timestep (500 noisy trajectories):')
print(f'{"t":>3} | {"Var(λ=0)":>10} | {"Var(λ=1)":>10} | {"ratio λ=1/λ=0":>14}')
print('-' * 50)
for t in range(T):
    v0 = advs_lam0[:, t].var()
    v1 = advs_lam1[:, t].var()
    print(f'{t:>3} | {v0:>10.4f} | {v1:>10.4f} | {v1/v0:>14.2f}x')

print('\n→ λ=1 (MC) variance is 4-5× higher than λ=0 (TD) at t=0')
print('→ 这就是 λ=0.95 实战推荐的根：在 bias 跟 variance 中取 sweet spot')

## 关键观察

1. **λ=0**：Â_t = δ_t（单步 TD），方差小但 bias 大（看不到长期 reward）
2. **λ=1**：Â_t = G_t - V(s_t)（MC advantage），方差大但 bias 0
3. **λ=0.5-0.95**：连续 bias-variance trade-off
4. **λ=0.95** 是 PPO/TRPO 实战默认——多数任务上经验最优

## Interview 启示

- 手撕 GAE: `Â_t = δ_t + γλ · Â_{t+1}`（reverse-time）+ `δ_t = r + γV(s') - V(s)`
- 解释 λ 是什么：bias-variance 旋钮，λ=0 全 TD，λ=1 全 MC
- 为什么 PPO 用 λ=0.95：高 λ 累积长期信号但方差大；0.95 是 OpenAI/DeepMind 实测最优

## 链回
- [[ch12-eligibility-traces]] §3.5
- [[ch13-reinforce-vs-baseline-cartpole.ipynb]] — 看 baseline 跟 GAE 是同源 variance 降低技术